In [ ]:
import sys, os
# ensure parent directory is on the path so `src` package can be imported
sys.path.insert(0, os.path.abspath('../..'))

In [ ]:
# configura per importare da src
import sys
sys.path.append('./src')

In [ ]:
# UNA TANTUM dopo aver scaricato dataset da NDA

import os
import tarfile
from glob import glob

# Assicurati che il percorso punti alla tua cartella con i dati scaricati
cartella_base = '../../OAI_Dataset/OAI_Baseline_XRays'

# Cerca ricorsivamente tutti i file .tar.gz
tar_files = glob(f"{cartella_base}/**/*.tar.gz", recursive=True)
print(f"Trovati {len(tar_files)} archivi compressi da estrarre.")

estratti_con_successo = 0
errori = 0

for tar_path in tar_files:
    try:
        # Apre l'archivio compresso in modalità lettura
        with tarfile.open(tar_path, "r:gz") as tar:
            # Estrae tutto il contenuto esattamente nella stessa directory in cui si trova il .tar.gz
            # Questo creerà automaticamente la sottocartella prevista dal manifest
            tar.extractall(path=os.path.dirname(tar_path))
        
        # ELIMINAZIONE ARCHIVIO (Consigliata)
        # Una volta estratto, il .tar.gz originale non serve più. 
        # Rimuovendolo libererai centinaia di GB sul tuo Google Drive.
        os.remove(tar_path)
        
        estratti_con_successo += 1
        
    except Exception as e:
        print(f"Errore durante l'estrazione di {os.path.basename(tar_path)}: {e}")
        errori += 1

print("\n--- OPERAZIONE COMPLETATA ---")
print(f"Archivi estratti e rimossi: {estratti_con_successo}")
if errori > 0:
    print(f"Attenzione: si sono verificati errori su {errori} archivi.")

In [ ]:
import pandas as pd
import re
from src.utils.dataset import VISUAL_CONCEPTS, TIMEPOINT_MAP, BASE_COLS_TO_KEEP, TARGET_COL
from src.utils.dataset import mappa_immagini_tramite_manifest, estrai_roi_dinamica, OAICBMDataset

FOLDER_PATH = '../../OAI_Dataset/SAS_Files'

df_list = []

for i in range(13):
    # Formatta l'iteratore a due cifre (00, 01, ..., 12)
    xx = f"{i:02d}"
    filename = f'kxr_sq_bu{xx}.sas7bdat'
    filepath = os.path.join(FOLDER_PATH, filename)

    if not os.path.exists(filepath):
        print(f"File non trovato: {filename}, salto...")
        continue

    print(f"Elaborazione di {filename}...")

    # 1. Lettura del file SAS
    df = pd.read_sas(filepath)

    # 2. Pulizia nomi colonne: minuscolo e rimozione prefissi temporali (v00, v01...)
    df.columns = [col.lower() for col in df.columns]
    df.columns = [re.sub(r'^v\d{2}', '', col) for col in df.columns]

    # 3. Decodifica robusta per l'ID (da byte string a stringa standard)
    if 'id' in df.columns:
        df['id'] = df['id'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x))

    if 'barcdbu' in df.columns:
        df['barcdbu'] = df['barcdbu'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x))

    # 4. Selezione delle colonne che ci interessano
    colonne_effettive = [col for col in BASE_COLS_TO_KEEP if col in df.columns]
    df_temp = df[colonne_effettive].copy()

    # 5. Filtraggio KLG e creazione chiave univoca immagine
    if TARGET_COL in df_temp.columns and 'side' in df_temp.columns:
        df_temp = df_temp.dropna(subset=[TARGET_COL])
        # Aggiungiamo anche da quale timepoint (XX) arrivano i dati, può fare comodo
        df_temp['timepoint'] = xx
        # La chiave primaria univoca per mappare poi l'immagine (ID_Lato_Timepoint)
        df_temp['image_key'] = df_temp['id'] + '_' + df_temp['side'].astype(int).astype(str) + '_' + xx

        df_list.append(df_temp)
    else:
        print(f"  -> {filename} ignorato: mancano colonne target o side essenziali.")

# 6. Concatenazione vettoriale finale
if df_list:
    df_pulito = pd.concat(df_list, ignore_index=True)
    print(f"\nOperazione completata! Dimensione totale del dataframe 'df_pulito': {df_pulito.shape}")
else:
    print("\nNessun dataframe valido trovato da concatenare.")

In [ ]:
df_pulito.head()

In [ ]:
df_pulito['timepoint'].unique()

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# ---------------------------------------------------------
# 1. TRASFORMAZIONI E PREPROCESSING IMMAGINE
# ---------------------------------------------------------
transform_pipeline = transforms.Compose([
    # Passando un singolo intero (224), PyTorch ridimensiona il lato PIÙ CORTO a 224
    # mantenendo intatte le proporzioni (l'immagine resterà un rettangolo alto).
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ---------------------------------------------------------
# 2. COLLEGAMENTO TRA METADATI (SAS) E IMMAGINI FISICHE
# ---------------------------------------------------------
ROOT_DIR = '../../OAI_Dataset'
MANIFEST_PATH = '../../OAI_Dataset/image03.txt'

# Applichiamo la funzione per scansionare le cartelle e tenere nel dataframe
# SOLO le annotazioni per le quali hai effettivamente l'immagine sul disco
df_finale = mappa_immagini_tramite_manifest(ROOT_DIR, df_pulito, MANIFEST_PATH)

# 1. Estraiamo la lista di tutti gli ID paziente univoci
pazienti_univoci = df_finale['id'].unique()
print(f"Totale pazienti clinici univoci: {len(pazienti_univoci)}")

# 2. Primo Split: Separiamo il Train (70%) dal resto (30%)
# Usiamo random_state=42 per garantire che lo split sia identico ad ogni avvio del notebook
pazienti_train, pazienti_temp = train_test_split(
    pazienti_univoci,
    test_size=0.30,
    random_state=42
)

# 3. Secondo Split: Dividiamo il restante 30% a metà (15% Validation, 15% Test)
pazienti_val, pazienti_test = train_test_split(
    pazienti_temp,
    test_size=0.50,
    random_state=42
)

print(f"Pazienti assegnati -> Train: {len(pazienti_train)} | Val: {len(pazienti_val)} | Test: {len(pazienti_test)}")

# 4. Popoliamo i DataFrame usando i gruppi di pazienti appena creati
# In questo modo, tutte le ginocchia (dx, sx, baseline, follow-up) seguono il loro paziente
df_train = df_finale[df_finale['id'].isin(pazienti_train)].copy()
df_val   = df_finale[df_finale['id'].isin(pazienti_val)].copy()
df_test  = df_finale[df_finale['id'].isin(pazienti_test)].copy()

print(f"\nRadiografie (Ginocchia) allocate -> Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

In [ ]:
# 5. Inizializziamo i 3 Dataset PyTorch (usando la classe OAICBMDataset che abbiamo creato)
train_dataset = OAICBMDataset(df_train, c_cols=VISUAL_CONCEPTS, y_col=TARGET_COL, transform=transform_pipeline)
val_dataset   = OAICBMDataset(df_val,   c_cols=VISUAL_CONCEPTS, y_col=TARGET_COL, transform=transform_pipeline)
test_dataset  = OAICBMDataset(df_test,  c_cols=VISUAL_CONCEPTS, y_col=TARGET_COL, transform=transform_pipeline)

# 6. Creiamo i DataLoader finali
# ATTENZIONE: shuffle=True si usa SOLO per il train set!
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("\nSplit completato! DataLoader pronti per il training loop.")

In [ ]:
from src.utils.dataset import salva_golden_dataset
import shutil

# 1. Avviamo la conversione per i tre split
salva_golden_dataset(train_dataset, 'train')
salva_golden_dataset(val_dataset, 'val')
salva_golden_dataset(test_dataset, 'test')

# 2. Creiamo un unico archivio ZIP dell'intera cartella
print("\nCompressione del Golden Dataset in corso (potrebbe richiedere qualche minuto)...")
shutil.make_archive('/content/OAI_Golden_Dataset', 'zip', '/content/OAI_Golden_Dataset')

# 3. Spostiamo il file ZIP finito sul tuo Google Drive per non perderlo alla chiusura di Colab
path_drive = '../../OAI_Dataset/OAI_BOX/OAI_Golden_Dataset.zip'
shutil.move('../../OAI_Dataset/OAI_Golden_Dataset.zip', path_drive)

print(f"\nOperazione completata! Il tuo Golden Dataset è al sicuro in: {path_drive}")